# `world_model`

This notebook loads trajectory windows, performs one training step, saves and reloads a checkpoint, and plans actions through the Python APIs. It runs on CPU with a tiny generated NPZ episode. Replace `DATA_PATH` with your own episode directory, local Parquet directory, or Hugging Face dataset ID to use real data.

Install the project with `pip install -e '.[dev]'` and open this notebook with that environment's Jupyter kernel. The import names are `world_model` and `utils` (from `src/`).

## Goals

1. Describe a world model as an **encoder**, an **action-conditioned dynamics model**, and **prediction heads**.
2. Recognize how synchronized trajectory windows become tensors for training.
3. Explain what the latent, state, and collision losses teach the model.
4. Use a learned model to compare candidate action sequences with CEM.

**Big picture:** at time $t$, observations $(\mathrm{RGB}, \mathrm{depth}, \mathrm{odom}, \mathrm{IMU})$ become a compact latent state $z_t$. Given an action $a_t$, dynamics predicts $\hat z_{t+1}=f(z_t,a_t)$. The state and risk heads decode useful quantities from $\hat z$. This lets a planner ask *what might happen if I take these actions?* without observing every hypothetical future.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from world_model import WorldModel, TrainConfig, rollout_loss, CEMConfig, cem_plan
from world_model.data import open_window_dataset, collate_for_dataset
from utils import load_checkpoint

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

### What is one training example?

A trajectory is a time-ordered record of camera images, depth, proprioception (odometry and IMU), actions, and sometimes collision labels. A **window** is a short, aligned slice of that record. Alignment matters: the model learns the consequence of the action paired with each observation, so shifted action timestamps teach the wrong transition. A batch has axes `B` (windows) and `T` (time); the remaining axes describe channels, image size, or sensor dimensions.

The project accepts NPZ episodes, local frame/window Parquet tables, and Hugging Face frame rows. `open_window_dataset` chooses the reader and `collate_for_dataset` chooses the matching collator. The separate `wm_datasets` package handles plain video clips and adjacent-frame pairs; those batches lack robot actions and state labels.

## 1. Open trajectory windows

`open_window_dataset` selects the NPZ, local Parquet, or Hugging Face reader. The example generates one episode with RGB, depth, IMU, odometry, actions, and collision labels. In a real project, set `DATASET_TYPE` to `"npz"` or `"hf"` and `DATA_PATH` to your data source. A local Parquet directory needs `frames.parquet` and `windows.parquet`; `"hf"` also accepts a Hub ID. Keep the image size consistent with the CNN constructor.

In [ ]:
scratch = TemporaryDirectory()
DATASET_TYPE = "npz"
DATA_PATH = str(Path(scratch.name) / "episodes")
IMAGE_SIZE = 32
HORIZON = 2

path = Path(DATA_PATH)
path.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(7)
t = 6
np.savez_compressed(
    path / "episode_000.npz",
    rgb=rng.integers(0, 256, (t, IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8),
    depth=rng.random((t, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32),
    imu=rng.normal(size=(t, 6)).astype("float32"),
    odom=rng.normal(size=(t, 6)).astype("float32"),
    action=rng.normal(size=(t, 3)).astype("float32"),
    collision=np.zeros(t, dtype="float32"),
)

dataset = open_window_dataset(
    dataset_type=DATASET_TYPE, path=DATA_PATH, split="train",
    image_size=IMAGE_SIZE, window_size=HORIZON + 1,
)
loader = DataLoader(dataset, batch_size=2, shuffle=True,
                    collate_fn=collate_for_dataset(dataset))
batch = next(iter(loader))
print("Windows:", len(dataset))
print("RGB:", tuple(batch.rgb.shape), "actions:", tuple(batch.action.shape))
print("Validity fields:", batch.depth_valid.shape, batch.collision_valid.shape)

### Visualize a window

The top row shows three aligned RGB and depth frames from one batch item. The lower plots show the action and odometry channels at those same time indices. Depth has its own color scale; its pixel values are not RGB intensities. The synthetic sensor traces have no physical meaning, but the shared time axis is the key training relationship.

In [ ]:
sample = 0
fig, axes = plt.subplots(4, HORIZON + 1, figsize=(10, 8),
                         gridspec_kw={"height_ratios": [1, 1, 0.8, 0.8]})
for step in range(HORIZON + 1):
    axes[0, step].imshow(batch.rgb[sample, step].permute(1, 2, 0).cpu())
    axes[0, step].set_title(f"RGB at t={step}")
    axes[1, step].imshow(batch.depth[sample, step, 0].cpu(), cmap="viridis")
    axes[1, step].set_title(f"Depth at t={step}")
    for row in (0, 1):
        axes[row, step].axis("off")
for row, values, label in (
    (2, batch.action[sample].cpu(), "Action channels"),
    (3, batch.odom[sample].cpu(), "Odometry channels"),
):
    ax = axes[row, 0]
    ax.plot(values.numpy(), marker="o")
    ax.set_xticks(range(HORIZON + 1))
    ax.set_xlabel("Time index")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
    for other in axes[row, 1:]:
        other.axis("off")
fig.suptitle("One synchronized trajectory window (synthetic fixture)")
fig.tight_layout()
plt.show()

The collator returns a `SampleBatch`: RGB is `(B,T,3,H,W)`, depth is `(B,T,1,H,W)`, and action, odometry, and IMU each have a final feature dimension. Validity masks identify missing optional sensor values. Use `dataset.D_odom`, `D_imu`, and `D_act` so the model follows the dataset schema.

### Inside the standard model

```text
RGB + depth ─► vision encoder ─┐
                               ├─► fused latent zₜ ─► dynamics(zₜ, aₜ) ─► predicted zₜ₊₁
odometry + IMU ─► proprio ──────┘                           │
                                               state head + collision-risk head
```

The CNN is a small baseline that can use RGB and depth. `dinov2` and `vjepa2` are alternative visual backbones and require their corresponding pretrained setup. The MLP dynamics predicts the next latent directly; RSSM carries a recurrent hidden state. `state_dim` controls how many odometry values the state head predicts. The code below uses all six generated odometry values; a real planner should choose dimensions that match its task and coordinate frame.

## 2. Construct and train a model

The default CNN consumes four channels (RGB plus depth). `rollout_loss` handles latent, state, and collision losses; its horizon requires at least `num_hist + horizon` frames. For a frozen pretrained encoder, choose `backbone="dinov2"` and the corresponding `backbone_kwargs` after installing its weights.

In [ ]:
model_kwargs = dict(
    odom_dim=dataset.D_odom,
    imu_dim=dataset.D_imu,
    act_dim=dataset.D_act,
    state_dim=dataset.D_odom,
    backbone="cnn",
    dynamics_type="mlp",  # "rssm" is also supported
    image_size=IMAGE_SIZE,
)
model = WorldModel(**model_kwargs).to(device)
cfg = TrainConfig(device=str(device), horizon=HORIZON, num_hist=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

model.train()
optimizer.zero_grad(set_to_none=True)
loss, metrics = rollout_loss(model, batch, cfg)
loss.backward()
optimizer.step()
print(metrics)

### Inspect the loss terms

These bars separate latent prediction, state prediction, and risk prediction before their configurable weights are applied. Their raw values have different units, so their heights alone do not show which term dominates the gradient. Use held-out data to judge whether training improves prediction.

In [ ]:
names = ["latent", "state", "risk"]
values = [metrics["loss_z"], metrics["loss_state"], metrics["loss_risk"]]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(names, values, color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_ylabel("Raw loss after one update")
ax.set_title("Training signals (synthetic fixture)")
fig.tight_layout()
plt.show()

### What does one optimizer step learn?

For a horizon of $H$, the model predicts $H$ future latent transitions from the first window observation and its actions. `rollout_loss` combines:

- **Latent loss:** mean squared error between predicted and encoded future states. The target encoding is detached, so this term trains prediction rather than chasing a moving target in the same backward pass.
- **State loss:** mean squared error on decoded odometry targets. This makes the latent useful for quantities a planner can score.
- **Risk loss:** binary cross entropy on collision labels. The batch's `collision_valid` mask keeps absent labels out of this term.

The weighted sum uses `TrainConfig.lambda_z`, `lambda_state`, and `lambda_risk`. With `objective="jepa-lite"`, the trainer uses latent matching without state and risk supervision; that changes what downstream planning can ask of the model. The values printed above are only a smoke check from random data.

## 2b. Repeat the update over a loader

The one-step cell exposes the API. In a training loop, each batch gets its own forward pass, backward pass, and optimizer step. The curve below is only a demonstration of how to collect metrics; the fixture is too small and random to support a scientific conclusion. For real training, evaluate on a separate validation split and save the best checkpoint using that metric.

In [ ]:
history = []
for epoch in range(3):
    model.train()
    epoch_losses = []
    for train_batch in loader:
        optimizer.zero_grad(set_to_none=True)
        step_loss, _ = rollout_loss(model, train_batch, cfg)
        step_loss.backward()
        optimizer.step()
        epoch_losses.append(step_loss.item())
    history.append(float(np.mean(epoch_losses)))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, len(history) + 1), history, "o-")
ax.set(xlabel="Epoch", ylabel="Mean training loss",
       title="Library training loop (synthetic fixture)")
ax.set_xticks(range(1, len(history) + 1))
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Save and reload

The loader in `utils` accepts a checkpoint dictionary with a `model` state dict. Use the same model arguments when restoring. A checkpoint from real training should also retain its model configuration and any normalization statistics used for training.

In [ ]:
checkpoint_path = Path(scratch.name) / "example.pt"
torch.save({"model": model.state_dict(), "model_kwargs": model_kwargs}, checkpoint_path)
restored = WorldModel(**model_kwargs).to(device)
report = load_checkpoint(restored, str(checkpoint_path), map_location=str(device), strict=True)
restored.eval()
print("Loaded parameters:", report["num_loaded"])

### From prediction to choice

A learned world model is useful because it can roll out *many* candidate action sequences from the same starting latent. The cross-entropy method (CEM) repeatedly samples sequences, scores their predicted outcomes, and refits a sampling distribution to the best candidates. In this lesson, the score is terminal distance to a goal plus average predicted collision risk.

CEM is **model predictive control** when the robot executes the first planned action, receives a new observation, and replans. This notebook plans once to show the API. In deployment, action bounds must reflect the robot's controls, and goal/state coordinates must match the model's training representation.

## 4. Encode, roll out, and plan

`encode` takes one observation step. `rollout` returns the initial latent plus one latent per action. `cem_plan` searches action sequences using a goal cost on predicted state and risk. The random fixture and single update produce meaningless motion predictions; inspect tensor shapes and API flow only.

In [ ]:
with torch.no_grad():
    observation = restored.encode(
        batch.rgb[:1, 0].to(device), batch.depth[:1, 0].to(device),
        batch.odom[:1, 0].to(device), batch.imu[:1, 0].to(device),
    )
    predicted_latents = restored.rollout(
        observation.z, batch.action[:1, :HORIZON].to(device),
    )
print("Latent rollout:", tuple(predicted_latents.shape))

# CEM uses the same action coordinates and state coordinates as the model.
def goal_cost(state_seq, risk_seq, goal):
    distance = torch.linalg.vector_norm(state_seq[:, -1, :2] - goal[:, :2], dim=-1)
    return distance + 0.2 * risk_seq[:, 1:].mean(dim=1)

plan_cfg = CEMConfig(horizon=HORIZON, iters=2, population=16,
                     elites=4, action_dim=dataset.D_act)
goal = torch.zeros(1, dataset.D_odom, device=device)
first_action, action_sequence = cem_plan(
    restored, observation.z, goal, goal_cost, plan_cfg, device,
)
print("First action:", first_action.shape,
      "planned sequence:", action_sequence.shape)

### See what the planner compares

The figure draws several sampled future trajectories and highlights CEM's selected sequence. It also marks the decoded starting position and goal. Since the model saw random data for one step, the paths are an **API illustration**, not useful forecasts. On a trained model, this view can reveal unrealistic jumps, goals in the wrong frame, or action bounds that are too wide.

In [ ]:
with torch.no_grad():
    candidate_actions = torch.randn(12, HORIZON, dataset.D_act, device=device)
    candidate_z = restored.rollout(observation.z.repeat(12, 1), candidate_actions)
    candidate_xy = restored.state_head(candidate_z)[..., :2].cpu().numpy()
    selected_z = restored.rollout(observation.z, action_sequence.unsqueeze(0))
    selected_xy = restored.state_head(selected_z)[0, :, :2].cpu().numpy()

fig, ax = plt.subplots(figsize=(5, 5))
for path_xy in candidate_xy:
    ax.plot(path_xy[:, 0], path_xy[:, 1], color="0.75", alpha=0.7)
ax.plot(selected_xy[:, 0], selected_xy[:, 1], "o-", color="#d62728",
        linewidth=2, label="CEM selected")
ax.scatter(*selected_xy[0], color="#1f77b4", s=70, label="Start")
ax.scatter(*goal[0, :2].cpu().numpy(), color="#2ca02c", marker="*",
           s=140, label="Goal")
ax.set(xlabel="Predicted state 0", ylabel="Predicted state 1",
       title="Candidate rollouts (synthetic fixture)")
ax.axis("equal")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

### Using real data

Change `DATASET_TYPE` and `DATA_PATH` in section 1 and remove the fixture creation. The same library call can open these sources:

```python
# Local NPZ episodes
open_window_dataset(dataset_type="npz", path="/data/episodes",
                    image_size=128, window_size=9)
# Local frames.parquet + windows.parquet directory
open_window_dataset(dataset_type="hf", path="/data/parquet_dataset",
                    split="train", image_size=128)
# Hugging Face frame rows (requires network access and dataset permission)
open_window_dataset(dataset_type="hf", path="owner/dataset",
                    split="train", image_size=128)
```

The collator remains `collate_for_dataset(dataset)`. When planning with a trained model, use its recorded preprocessing, normalization statistics, action bounds, and coordinate frame; the defaults above are only a shape example. See [world model guide](../world_model.md) for the architecture and [video dataset notebook](video_datasets.ipynb) for `wm_datasets` video loaders.

For the architectural variants and longer training runs, see the [world model guide](../world_model.md) and [training guide](../training.md).